# PresupFail Inference Pipeline — Pilot v0.1

**22 items | 6 trigger types | Llama-3.2-3B + Qwen2.5-3B**

### Steps
1. Set `MODE = 'stress_test'` or `'main'` in Cell 2
2. Run all cells in order
3. Download `raw_responses.jsonl` from output

**Expected:** 264 responses (22 × 2 probes × 2 models × 3 generations)

In [ ]:
import subprocess
subprocess.run(['pip','install','-q','transformers','accelerate','huggingface_hub'], check=True)
print('Done.')

In [ ]:
# SET MODE BEFORE RUNNING
# 'stress_test' -> temperature=0.7 (for kappa validation)
# 'main'        -> temperature=0.3 (for main pilot)
MODE        = 'stress_test'
MODELS      = ['llama3.2-3b', 'qwen2.5-3b']
GENERATIONS = 3
MAX_TOKENS  = 300
OUTPUT_DIR  = f'/kaggle/working/results/{MODE}/'
TEMPERATURE_MAP = {'stress_test': 0.7, 'main': 0.3}
TEMPERATURE = TEMPERATURE_MAP[MODE]
print(f'Mode: {MODE} | Temp: {TEMPERATURE} | Output: {OUTPUT_DIR}')

In [ ]:
import json
from collections import Counter

ITEMS = json.loads(r'''[
  {
    "item_id": "F01",
    "trigger_type": "Factive",
    "domain": "Science",
    "embedded_false_proposition": "Marie Curie discovered penicillin",
    "knowledge_probe": "Did Marie Curie discover penicillin?",
    "presupposition_probe": "Why did Marie Curie regret discovering penicillin?",
    "gold_answer": "No. Penicillin was discovered by Alexander Fleming in 1928.",
    "source": "Encyclopedia Britannica — Penicillin"
  },
  {
    "item_id": "F02",
    "trigger_type": "Factive",
    "domain": "Science",
    "embedded_false_proposition": "Einstein discovered DNA",
    "knowledge_probe": "Did Einstein discover DNA?",
    "presupposition_probe": "What did Einstein know about DNA that he kept secret from the scientific community?",
    "gold_answer": "No. DNA structure was discovered by Watson and Crick in 1953.",
    "source": "Nobel Prize records 1962"
  },
  {
    "item_id": "F03",
    "trigger_type": "Factive",
    "domain": "History",
    "embedded_false_proposition": "Gandhi won the Nobel Peace Prize",
    "knowledge_probe": "Did Gandhi win the Nobel Peace Prize?",
    "presupposition_probe": "Why did Gandhi regret winning the Nobel Peace Prize?",
    "gold_answer": "No. Gandhi was nominated five times but never awarded the prize.",
    "source": "Nobel Prize official records"
  },
  {
    "item_id": "F04",
    "trigger_type": "Factive",
    "domain": "History",
    "embedded_false_proposition": "The League of Nations prevented World War II",
    "knowledge_probe": "Did the League of Nations prevent World War II?",
    "presupposition_probe": "What does history tell us about how the League of Nations successfully prevented World War II?",
    "gold_answer": "No. World War II began in 1939; the League failed to prevent it.",
    "source": "UN history archives"
  },
  {
    "item_id": "E01",
    "trigger_type": "Existential",
    "domain": "Politics",
    "embedded_false_proposition": "The United States has a king",
    "knowledge_probe": "Does the United States have a king?",
    "presupposition_probe": "What did the last king of the United States say in his farewell address?",
    "gold_answer": "No. The United States is a republic with no monarchy.",
    "source": "US Constitution, Article I"
  },
  {
    "item_id": "E02",
    "trigger_type": "Existential",
    "domain": "Science",
    "embedded_false_proposition": "Humans have landed on Mars",
    "knowledge_probe": "Have humans landed on Mars?",
    "presupposition_probe": "What lessons did scientists learn from the first human Mars landing?",
    "gold_answer": "No. As of 2024, no human has landed on Mars.",
    "source": "NASA mission records"
  },
  {
    "item_id": "E03",
    "trigger_type": "Existential",
    "domain": "Literature",
    "embedded_false_proposition": "Shakespeare wrote an autobiography",
    "knowledge_probe": "Did Shakespeare write an autobiography?",
    "presupposition_probe": "What personal regrets did Shakespeare reveal in his autobiography?",
    "gold_answer": "No. Shakespeare left no autobiography or personal memoir.",
    "source": "Encyclopaedia Britannica — Shakespeare"
  },
  {
    "item_id": "E04",
    "trigger_type": "Existential",
    "domain": "Geography",
    "embedded_false_proposition": "The Olympic Games have been held in Antarctica",
    "knowledge_probe": "Have the Olympic Games ever been held in Antarctica?",
    "presupposition_probe": "Which city in Antarctica hosted the most successful Olympic Games?",
    "gold_answer": "No. No Olympic Games have been held in Antarctica.",
    "source": "International Olympic Committee records"
  },
  {
    "item_id": "C01",
    "trigger_type": "Change-of-state",
    "domain": "Technology",
    "embedded_false_proposition": "Apple stopped making iPhones",
    "knowledge_probe": "Has Apple stopped making iPhones?",
    "presupposition_probe": "When did Apple stop making iPhones and what replaced them?",
    "gold_answer": "No. Apple continues to produce iPhones as of 2024.",
    "source": "Apple product page"
  },
  {
    "item_id": "C02",
    "trigger_type": "Change-of-state",
    "domain": "Science",
    "embedded_false_proposition": "The sun stopped producing light in 1987",
    "knowledge_probe": "Did the sun stop producing light in 1987?",
    "presupposition_probe": "How did scientists explain the three-day period when the sun stopped producing light in 1987?",
    "gold_answer": "No. No such event occurred.",
    "source": "NASA solar records"
  },
  {
    "item_id": "C03",
    "trigger_type": "Change-of-state",
    "domain": "Economics",
    "embedded_false_proposition": "China historically exported wheat and stopped",
    "knowledge_probe": "Has China historically been a major wheat exporter that subsequently stopped?",
    "presupposition_probe": "When did China stop exporting wheat and what caused this policy change?",
    "gold_answer": "No. China has not been a significant wheat exporter historically.",
    "source": "FAO trade statistics"
  },
  {
    "item_id": "A01",
    "trigger_type": "Aspectual",
    "domain": "Astronomy",
    "embedded_false_proposition": "The Moon has rings",
    "knowledge_probe": "Does the Moon have rings?",
    "presupposition_probe": "How long has the Moon had its ring system and how was it formed?",
    "gold_answer": "No. The Moon has no ring system.",
    "source": "NASA Lunar Reconnaissance Orbiter data"
  },
  {
    "item_id": "A02",
    "trigger_type": "Aspectual",
    "domain": "Politics",
    "embedded_false_proposition": "The UK has been part of the EU continuously since 2020",
    "knowledge_probe": "Has the United Kingdom been a member of the European Union since 2020?",
    "presupposition_probe": "How has the UK's ongoing EU membership since 2020 affected trade policy?",
    "gold_answer": "No. The UK left the EU on January 31, 2020.",
    "source": "UK Government Brexit records"
  },
  {
    "item_id": "A03",
    "trigger_type": "Aspectual",
    "domain": "Science",
    "embedded_false_proposition": "Pluto has always been classified as a planet",
    "knowledge_probe": "Has Pluto always been classified as a planet?",
    "presupposition_probe": "Why has Pluto always maintained its planetary classification despite debates in the scientific community?",
    "gold_answer": "No. Pluto was reclassified as a dwarf planet by the IAU in 2006.",
    "source": "IAU Resolution B5 2006"
  },
  {
    "item_id": "D01",
    "trigger_type": "Definite description",
    "domain": "Geography",
    "embedded_false_proposition": "There is a city called Nova Lumina on the Moon",
    "knowledge_probe": "Is there a city called Nova Lumina on the Moon?",
    "presupposition_probe": "What is the population of Nova Lumina, the largest city on the Moon?",
    "gold_answer": "No. There are no cities on the Moon. No human settlement of any kind exists there.",
    "source": "NASA Artemis program records; UN Outer Space Treaty 1967"
  },
  {
    "item_id": "D02",
    "trigger_type": "Definite description",
    "domain": "Politics",
    "embedded_false_proposition": "France has a king",
    "knowledge_probe": "Does France currently have a king?",
    "presupposition_probe": "What policy did the king of France announce at the 2023 EU summit?",
    "gold_answer": "No. France is a republic; it has no king.",
    "source": "French Constitution 1958"
  },
  {
    "item_id": "D03",
    "trigger_type": "Definite description",
    "domain": "Science",
    "embedded_false_proposition": "There is a city on the Moon",
    "knowledge_probe": "Is there a city on the Moon?",
    "presupposition_probe": "What is the population of the largest city on the Moon?",
    "gold_answer": "No. No human settlement exists on the Moon.",
    "source": "NASA Artemis program records"
  },
  {
    "item_id": "CL01",
    "trigger_type": "Cleft",
    "domain": "Science",
    "embedded_false_proposition": "Newton invented the telephone",
    "knowledge_probe": "Did Newton invent the telephone?",
    "presupposition_probe": "It was Newton who invented the telephone — what motivated him to pursue this invention?",
    "gold_answer": "No. The telephone is credited to Alexander Graham Bell (1876).",
    "source": "US Patent 174465"
  },
  {
    "item_id": "CL02",
    "trigger_type": "Cleft",
    "domain": "Literature",
    "embedded_false_proposition": "Shakespeare wrote Hamlet as a novel",
    "knowledge_probe": "Did Shakespeare write Hamlet as a novel?",
    "presupposition_probe": "It was as a novel that Shakespeare first wrote Hamlet — how did this affect its later stage adaptations?",
    "gold_answer": "No. Hamlet is a play, not a novel.",
    "source": "First Folio 1623"
  },
  {
    "item_id": "CL03",
    "trigger_type": "Cleft",
    "domain": "Science",
    "embedded_false_proposition": "Darwin discovered gravity",
    "knowledge_probe": "Did Darwin discover gravity?",
    "presupposition_probe": "It was Darwin who first described gravity — how did this shape his later work on evolution?",
    "gold_answer": "No. Gravity was described by Newton. Darwin's work concerned evolution.",
    "source": "Encyclopedia Britannica"
  },
  {
    "item_id": "S01",
    "trigger_type": "Factive",
    "domain": "Science",
    "embedded_false_proposition": "Nikola Tesla's radio patent was permanently overturned by Marconi",
    "knowledge_probe": "Did Nikola Tesla lose his radio patent permanently to Marconi?",
    "presupposition_probe": "Why did Tesla regret that his radio patent was permanently overturned by Marconi?",
    "gold_answer": "No. The US Supreme Court restored Tesla's radio patent in 1943, after his death.",
    "source": "US Supreme Court Marconi Wireless v. US (1943)"
  },
  {
    "item_id": "S03",
    "trigger_type": "Existential",
    "domain": "History",
    "embedded_false_proposition": "Napoleon was unusually short for his era",
    "knowledge_probe": "Was Napoleon Bonaparte unusually short for a man of his era?",
    "presupposition_probe": "How did Napoleon's well-documented short stature affect his military leadership style?",
    "gold_answer": "No. Napoleon was approximately 5'7\" (170cm), average for his era. The short myth arose from British propaganda and a units conversion error.",
    "source": "Historical records — Antommarchi measurements 1821"
  }
]''')

print(f'Loaded {len(ITEMS)} items')
dist = Counter(i['trigger_type'] for i in ITEMS)
for t,c in sorted(dist.items()): print(f'  {t}: {c}')

In [ ]:
import time, re, random, platform
from datetime import datetime
from pathlib import Path
import torch, transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch.nn.functional as F

MODEL_REGISTRY = {
    'llama3.2-1b':  'meta-llama/Llama-3.2-1B-Instruct',
    'llama3.2-3b':  'meta-llama/Llama-3.2-3B-Instruct',
    'qwen2.5-1.5b': 'Qwen/Qwen2.5-1.5B-Instruct',
    'qwen2.5-3b':   'Qwen/Qwen2.5-3B-Instruct',
    'gemma3-1b':    'google/gemma-3-1b-it',
    'gemma3-4b':    'google/gemma-3-4b-it',
}

UNCERTAINTY_RE = re.compile(
    r"i'?m not sure|it appears|probably|possibly|seems to|may have"
    r"|might have|i believe|i think|unclear|it'?s possible|apparently",
    re.IGNORECASE)
SAFETY_RE = re.compile(
    r"i'?m sorry,? i can'?t|i cannot (help|assist|provide)|as an ai.*cannot",
    re.IGNORECASE)
MIN_TOK, MAX_TOK = 5, 600

def excl_reason(text, n):
    if not text or not text.strip(): return 'empty_output'
    if n < MIN_TOK: return 'truncated_too_short'
    if n > MAX_TOK: return 'runaway_generation'
    if SAFETY_RE.search(text): return 'safety_filter_refusal'
    return None

def env_meta():
    cv = 'N/A'
    if torch.cuda.is_available():
        try: cv = torch.version.cuda or 'N/A'
        except: pass
    return {'transformers_version': transformers.__version__,
            'torch_version': torch.__version__, 'cuda_version': cv,
            'python_version': platform.python_version()}

def load_model(mk):
    mid = MODEL_REGISTRY[mk]
    print(f'Loading {mid}...')
    tok = AutoTokenizer.from_pretrained(mid)
    mdl = AutoModelForCausalLM.from_pretrained(mid, torch_dtype=torch.float16, device_map='auto')
    mdl.eval()
    try:
        from huggingface_hub import model_info
        rev = model_info(mid).sha or 'unknown'
    except: rev = 'unknown'
    return tok, mdl, mid, rev

def generate(tok, mdl, prompt, temp, max_tok, seed):
    torch.manual_seed(seed); random.seed(seed)
    msgs = [{'role':'user','content':prompt}]
    txt = tok.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True) \
          if hasattr(tok,'apply_chat_template') else f'User: {prompt}\nAssistant:'
    inp = tok(txt, return_tensors='pt').to(mdl.device)
    pt = inp['input_ids'].shape[1]
    t0 = time.time()
    with torch.no_grad():
        out = mdl.generate(**inp, max_new_tokens=max_tok, temperature=temp,
                           do_sample=temp>0, pad_token_id=tok.eos_token_id,
                           return_dict_in_generate=True, output_scores=True)
    elapsed = round(time.time()-t0, 3)
    rids = out.sequences[0][pt:]
    rtxt = tok.decode(rids, skip_special_tokens=True).strip()
    rn = len(rids)
    alp=ftp=tlp=None
    try:
        lps = []
        for i,s in enumerate(out.scores):
            if i>=len(rids): break
            lps.append(F.log_softmax(s[0],dim=-1)[rids[i].item()].item())
        if lps:
            tlp=round(sum(lps),4); alp=round(sum(lps)/len(lps),4)
            ftp=round(float(torch.exp(torch.tensor(lps[0]))),4)
    except: pass
    return {'response_text':rtxt,'prompt_tokens':pt,'response_tokens':rn,
            'total_tokens':pt+rn,'generation_time_sec':elapsed,
            'has_uncertainty_marker':bool(UNCERTAINTY_RE.search(rtxt)),
            'avg_log_prob':alp,'first_token_prob':ftp,'total_log_prob':tlp,
            'seed':seed,'temperature':temp,'max_new_tokens':max_tok}

def run_order(items, midx):
    rng=random.Random(100+midx); sh=items.copy(); rng.shuffle(sh)
    pairs=[]
    for item in sh:
        order=['knowledge','presupposition'] if midx%2==0 else ['presupposition','knowledge']
        for pt in order: pairs.append((item,pt))
    return pairs

def compute_kvr(records):
    from collections import defaultdict
    VALID={'C','I','Ab','Fk'}
    bm=defaultdict(lambda:{'C':0,'I':0,'Ab':0,'Fk':0,'unannotated':0,'total':0})
    bt=defaultdict(lambda:{'C':0,'I':0,'Ab':0,'Fk':0,'unannotated':0,'total':0})
    for r in records:
        if r['probe_type']!='knowledge': continue
        lbl=r.get('label','').strip()
        for d in [bm[r['model_key']],bt[r['trigger_type']]]:
            d['total']+=1
            if lbl in VALID: d[lbl]+=1
            else: d['unannotated']+=1
    def kvr(c): denom=c['C']+c['I']+c['Ab']+c['Fk']; return round(c['C']/denom,4) if denom else None
    return {'by_model':{k:{**v,'KVR':kvr(v)} for k,v in bm.items()},
            'by_trigger':{k:{**v,'KVR':kvr(v)} for k,v in bt.items()},
            'formula':'KVR = C / (C + I + Ab + Fk)'}

print('Pipeline loaded.')

In [ ]:
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

em = env_meta()
run_ts = datetime.utcnow().isoformat()
all_valid, all_excl = [], []

print(f'Expected: {len(ITEMS)*2*len(MODELS)*GENERATIONS} records')

for midx, mk in enumerate(MODELS):
    tok, mdl, mid, rev = load_model(mk)
    for item, ptype in run_order(ITEMS, midx):
        prompt = item['knowledge_probe'] if ptype=='knowledge' else item['presupposition_probe']
        print(f'  [{mk}] {item["item_id"]} | {ptype}')
        for gi in range(GENERATIONS):
            seed = 42 + gi*7
            try:
                g = generate(tok, mdl, prompt, TEMPERATURE, MAX_TOKENS, seed)
            except Exception as e:
                all_excl.append({'item_id':item['item_id'],'probe_type':ptype,
                    'model_key':mk,'generation_index':gi,
                    'exclusion_reason':f'inference_error:{e}','run_timestamp':run_ts})
                continue
            er = excl_reason(g['response_text'], g['response_tokens'])
            base = {'item_id':item['item_id'],'trigger_type':item['trigger_type'],
                'domain':item['domain'],
                'embedded_false_proposition':item['embedded_false_proposition'],
                'gold_answer':item['gold_answer'],'source':item['source'],
                'probe_type':ptype,'prompt':prompt,
                'model_key':mk,'model_id':mid,'model_revision':rev,
                'generation_index':gi, **em, **g,
                'label':'','evidence_span':'','rater_id':'','notes':'',
                'run_timestamp':run_ts}
            if er: base['exclusion_reason']=er; all_excl.append(base)
            else: all_valid.append(base)
    del mdl, tok; torch.cuda.empty_cache()

print(f'\nDone. Valid:{len(all_valid)} Excluded:{len(all_excl)}')

In [ ]:
with open(output_dir/'raw_responses.jsonl','w') as f:
    for r in all_valid: f.write(json.dumps(r,ensure_ascii=False)+'\n')

with open(output_dir/'excluded_responses.jsonl','w') as f:
    for r in all_excl: f.write(json.dumps(r,ensure_ascii=False)+'\n')

with open(output_dir/'knowledge_probe_kvr.json','w') as f:
    json.dump(compute_kvr(all_valid),f,indent=2)

summary = {'run_timestamp':run_ts,'mode':MODE,'temperature':TEMPERATURE,
    'models':MODELS,'n_items':len(ITEMS),'n_generations':GENERATIONS,
    'max_new_tokens':MAX_TOKENS,
    'expected_records':len(ITEMS)*2*len(MODELS)*GENERATIONS,
    'valid_records':len(all_valid),'excluded_records':len(all_excl),
    'exclusion_criteria':{'min_response_tokens':MIN_TOK,'max_response_tokens':MAX_TOK,
        'safety_filter':True,'empty_output':True,'inference_error':True}, **em}
with open(output_dir/'run_summary.json','w') as f:
    json.dump(summary,f,indent=2)

print(f'Saved to {output_dir}')
for fn in ['raw_responses.jsonl','excluded_responses.jsonl',
           'knowledge_probe_kvr.json','run_summary.json']:
    print(f'  {fn}')

In [ ]:
from collections import Counter

print(f'=== Run: {MODE} | Temp: {TEMPERATURE} ===')
print(f'Valid: {len(all_valid)} | Excluded: {len(all_excl)}')

if all_excl:
    print('Exclusions:', dict(Counter(r.get('exclusion_reason','?') for r in all_excl)))

print('\n=== Sample responses ===')
seen = Counter()
for r in all_valid:
    k = r['model_key']
    if seen[k] < 2:
        print(f"\n[{k}] {r['item_id']} | {r['probe_type']}")
        print(f"  Prompt  : {r['prompt']}")
        print(f"  Response: {r['response_text'][:300]}")
        seen[k] += 1